# Tunix-Med: Baseline Medical Knowledge Evaluation

Measures how well the **pre-trained `google/gemma-3-1b-it` model** answers
cardiology questions **before any fine-tuning**.  The scores produced here
are the **baseline** to compare against after Tunix domain-specific training.

### Evaluation pipeline

For each question the notebook:
1. Generates an answer using the pre-trained model (no adapter loaded).
2. Computes **perplexity** of the reference answer.
3. Scores the answer with three independent methods:
   - **Keyword matching**
   - **Semantic similarity** (Sentence-BERT `all-MiniLM-L6-v2`)
   - **AI judge** (Qwen2.5-7B-Instruct in 4-bit)
4. Saves results to `medical_baseline_results.csv` for comparison.

> **Important:** The questions are sampled from the held-out split of
> `lmassaron/medical-cardiology-qa` (seed=42, 10% eval) — the **same 25
> questions** used in the fine-tuned evaluation notebook, so the two sets of
> scores are directly comparable.

## 1 · Setup & Model Loading

In [1]:
import os, warnings, re
import numpy as np
import torch
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer, util
from huggingface_hub import login

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

token = os.environ.get("HF_TOKEN")
if token:
    login(token=token)


def info_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


device = info_device()
# Gemma 3 overflows float16; use bfloat16 on Ampere+ GPUs, float32 otherwise.
dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
    else torch.float32
)
print(f"Device : {device}  |  dtype : {dtype}")

MODEL_NAME = "google/gemma-3-1b-it"
print("Loading base model (no adapter) ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=dtype,  # correct kwarg for current Transformers
    device_map=device,
)
model.eval()
print("Model ready.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Device : cuda  |  dtype : torch.bfloat16
Loading base model (no adapter) ...
Model ready.


## 2 · Evaluation Dataset

We load the held-out questions from `lmassaron/medical-cardiology-qa` using
the **identical** seed, split fraction, and sampling logic as the fine-tuned
evaluation notebook.  This guarantees the baseline and fine-tuned scores are
measured on exactly the same questions.

In [2]:
DATASET_ID = "lmassaron/medical-cardiology-qa"
EVAL_SPLIT = 0.1
SEED = 42
N_EVAL_QS = 300

print(f"Loading {DATASET_ID} ...")
full_ds = load_dataset(DATASET_ID, split="train")
n = len(full_ds)

rng = np.random.default_rng(SEED)
all_idx = rng.permutation(n)
cut = int(n * (1.0 - EVAL_SPLIT))
eval_idx = all_idx[cut:]

print(f"  Total rows : {n:,}  |  Eval rows : {len(eval_idx):,}")


def extract_qa(example):
    msgs = example["messages"]
    return {
        "question": next(m["content"] for m in msgs if m["role"] == "user"),
        "answer": next(m["content"] for m in msgs if m["role"] == "assistant"),
    }


rng2 = np.random.default_rng(SEED + 1)
shuffled = rng2.permutation(eval_idx)

seen_prefixes, qa_pairs = set(), []
for idx in shuffled:
    if len(qa_pairs) >= N_EVAL_QS:
        break
    ex = extract_qa(full_ds[int(idx)])
    q, a = ex["question"], ex["answer"]
    if len(a) < 25:
        continue
    prefix = " ".join(q.lower().split()[:4])
    if prefix in seen_prefixes:
        continue
    seen_prefixes.add(prefix)
    qa_pairs.append({"question": q, "answer": a, "dataset_idx": int(idx)})

data = pd.DataFrame(qa_pairs)
print(f"\nSampled {len(data)} evaluation questions")
print("\nSample questions:")
for _, row in data.head(5).iterrows():
    print(f"  Q: {row['question'][:90]}")
    print(f"  A: {row['answer'][:80]}")
    print()

Loading lmassaron/medical-cardiology-qa ...
  Total rows : 10,518  |  Eval rows : 1,052

Sampled 300 evaluation questions

Sample questions:
  Q: What is the rate of bleeding stroke associated with statin use over 5 years of treatment p
  A: Over 5 years of treatment, statins result in 7.5 cases of bleeding stroke per 10

  Q: Which medications used in cardiac stress testing can cause mild hypotension?
  A: Adenosine and dipyridamole.

  Q: What is the advantage of using perfusion stress test with 99mTc labelled sestamibi?
  A: It is appropriate for select patients with an abnormal resting electrocardiogram

  Q: What is the suggested course of action if angina is suspected?
  A: An urgent medical assessment is suggested to rule out serious medical conditions

  Q: What is the benefit of using beta blockers in glaucoma treatment?
  A: They can lower intraocular pressure.



## 3 · Baseline Inference Loop

We use the **same system prompt** that was used during fine-tuning so the
prompt format is identical between baseline and fine-tuned evaluations.
Without this, the comparison is unfair — a different prompt shifts the output
distribution even before any training effect.

In [3]:
# Must match the system prompt in the training data exactly
SYSTEM_PROMPT = (
    "You are a knowledgeable medical assistant specializing in cardiology. "
    "Answer clinical questions accurately, focusing on diagnostic criteria, "
    "treatment guidelines, and pathophysiology."
)

results_list = []
model.eval()

for _, row in tqdm(data.iterrows(), total=len(data)):
    question = row["question"]
    answer = row["answer"]

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    )
    input_ids = encoded.to(device)
    attention_mask = torch.ones_like(
        input_ids
    )  # explicit mask avoids pad-token ambiguity

    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,
        )

    gen_ids = outputs[0, input_ids.shape[-1] :]
    generated = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

    # Reference perplexity: how surprised the *base* model is by the correct answer
    ref_ids = tokenizer(
        answer, return_tensors="pt", add_special_tokens=False
    ).input_ids.to(device)
    full_ids = torch.cat([input_ids, ref_ids], dim=1)
    labels = full_ids.clone()
    labels[:, : input_ids.shape[1]] = -100
    with torch.no_grad():
        loss = model(
            full_ids, attention_mask=torch.ones_like(full_ids), labels=labels
        ).loss
    perplexity = torch.exp(loss).item()

    results_list.append(
        {
            "question": question,
            "expected_answer": answer,
            "generated_answer": generated,
            "perplexity": perplexity,
        }
    )

results_df = pd.DataFrame(results_list)
print(f"Generated {len(results_df)} answers")
results_df[["question", "expected_answer", "generated_answer", "perplexity"]].head()

100%|██████████| 300/300 [22:36<00:00,  4.52s/it]

Generated 300 answers


,question,expected_answer,generated_answer,perplexity
0,What is the rate of bleeding stroke associated...,"Over 5 years of treatment, statins result in 7...","Okay, let’s tackle this question about the rat...",27.679310
1,Which medications used in cardiac stress testi...,Adenosine and dipyridamole.,"Okay, let’s delve into the medications that ca...",2712.877197
2,What is the advantage of using perfusion stres...,It is appropriate for select patients with an ...,"Okay, let’s delve into the advantages of the P...",67170.296875
3,What is the suggested course of action if angi...,An urgent medical assessment is suggested to r...,"Okay, let’s talk about managing suspected angi...",12280.543945
4,What is the benefit of using beta blockers in ...,They can lower intraocular pressure.,"Okay, let’s delve into the fascinating and som...",774856.687500


## 4 · Scoring Metrics

Three independent metrics, combined into one weighted final score
(keyword 20 %, semantic 40 %, AI judge 40 %).

In [4]:
# ── A. Keyword overlap ───────────────────────────────────────────────────────
def calculate_keyword_score(generated, expected):
    kws = set(re.findall(r"\b\w{5,}\b", expected.lower()))
    if not kws:
        return 1.0
    return sum(1 for k in kws if k in generated.lower()) / len(kws)


# ── B. Semantic similarity (Sentence-BERT) ───────────────────────────────────
similarity_model = SentenceTransformer("all-MiniLM-L6-v2")


def calculate_semantic_similarity(generated, expected):
    e1 = similarity_model.encode(generated, convert_to_tensor=True)
    e2 = similarity_model.encode(expected, convert_to_tensor=True)
    return float(util.pytorch_cos_sim(e1, e2))


# ── C. AI judge (Qwen2.5-7B-Instruct in 4-bit) ───────────────────────────────
JUDGE_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
print(f"Loading AI judge: {JUDGE_MODEL_NAME} ...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)
judge_tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL_NAME)
judge_model = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)


def ai_judge_score(question, generated, expected):
    prompt = (
        "You are a medical examiner.\n"
        "Rate the generated answer vs the reference on a 1-10 scale.\n\n"
        f"Question: {question}\n"
        f"Reference: {expected}\n"
        f"Generated: {generated}\n\n"
        "10=perfect, 7-9=mostly correct, 4-6=partial, 1-3=wrong/misleading.\n"
        "Return ONLY the number."
    )
    inp = judge_tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(judge_model.device)
    mask = torch.ones_like(inp)
    with torch.no_grad():
        out = judge_model.generate(
            inp, attention_mask=mask, max_new_tokens=5, do_sample=False
        )
        txt = judge_tokenizer.decode(
            out[0, inp.shape[-1] :], skip_special_tokens=True
        ).strip()
    m = re.search(r"\d+", txt)
    return int(m.group()) / 10.0 if m else 0.5

Loading AI judge: Qwen/Qwen2.5-7B-Instruct ...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

## 5 · Results & Baseline Summary

In [5]:
print("Scoring baseline results...")
tqdm.pandas()

results_df["keyword_score"] = results_df.progress_apply(
    lambda r: calculate_keyword_score(r["generated_answer"], r["expected_answer"]),
    axis=1,
)
results_df["semantic_score"] = results_df.progress_apply(
    lambda r: calculate_semantic_similarity(
        r["generated_answer"], r["expected_answer"]
    ),
    axis=1,
)
results_df["ai_judge_score"] = results_df.progress_apply(
    lambda r: ai_judge_score(
        r["question"], r["generated_answer"], r["expected_answer"]
    ),
    axis=1,
)

results_df["final_score"] = (
    results_df["keyword_score"] * 0.2
    + results_df["semantic_score"] * 0.4
    + results_df["ai_judge_score"] * 0.4
)

# Save for comparison with the fine-tuned notebook
results_df.to_csv("medical_baseline_results.csv", index=False)
print("Saved → medical_baseline_results.csv")

print("\n─── Baseline Performance Summary ──────────────────────────────────")
print(f"  Keyword  score (mean): {results_df['keyword_score'].mean():.3f}")
print(f"  Semantic score (mean): {results_df['semantic_score'].mean():.3f}")
print(f"  AI judge score (mean): {results_df['ai_judge_score'].mean():.3f}")
print(f"  Final    score (mean): {results_df['final_score'].mean():.3f}")
print(f"  Perplexity     (mean): {results_df['perplexity'].mean():.1f}")
print("────────────────────────────────────────────────────────────────────")

print("\n── Top 3 baseline answers ──")
for _, r in results_df.nlargest(3, "final_score").iterrows():
    print(f"  [{r['final_score']:.2f}]  Q: {r['question'][:70]}")
    print(f"         Expected : {r['expected_answer'][:70]}")
    print(f"         Generated: {r['generated_answer'][:70]}")
    print()

print("── Bottom 3 baseline answers ──")
for _, r in results_df.nsmallest(3, "final_score").iterrows():
    print(f"  [{r['final_score']:.2f}]  Q: {r['question'][:70]}")
    print(f"         Expected : {r['expected_answer'][:70]}")
    print(f"         Generated: {r['generated_answer'][:70]}")
    print()

Scoring baseline results...


100%|██████████| 300/300 [01:42<00:00,  2.93it/s]

Saved → medical_baseline_results.csv

─── Baseline Performance Summary ──────────────────────────────────
  Keyword  score (mean): 0.415
  Semantic score (mean): 0.520
  AI judge score (mean): 0.662
  Final    score (mean): 0.556
  Perplexity     (mean): 30609236.4
────────────────────────────────────────────────────────────────────

── Top 3 baseline answers ──
  [0.84]  Q: What is a ventricular septal defect (VSD)?
         Expected : A ventricular septal defect (VSD) is a defect in the ventricular septu
         Generated: Okay, let’s dive into Ventricular Septal Defects – VSDs. As a cardiolo

  [0.82]  Q: What is third-degree AV block also known as?
         Expected : Third-degree AV block is also known as complete heart block.
         Generated: Okay, let’s tackle this. “Third-degree AV block” – it's a very specifi

  [0.82]  Q: What condition can be diagnosed using coronary catheterization?
         Expected : Coronary catheterization can diagnose coronary artery disease and my

## Conclusion

The scores above represent the **untrained capability** of `gemma-3-1b-it`
on Wikipedia-derived cardiology factual Q&A, using the same system prompt
and questions as the fine-tuned evaluation.

**What to expect after fine-tuning:**
- Lower perplexity — the model becomes less surprised by correct factual answers.
- Higher keyword and semantic scores — generated answers contain more of the
  domain-specific terms and phrasings present in the training data.
- Higher AI judge scores — responses are more factually accurate and concise.

Run `04_medical_evaluation_final.ipynb` after training to see the delta.